# Семинар 1. Первая прикладная линейная регрессия

## Можем ли мы предсказывать стоимость дома лучше, чем просто называть среднюю цену?

На предыдущем семинаре мы сами обучали линейную модель, уменьшая MSE с помощью градиентного спуска. Теперь применим знакомую идею к реальным данным и познакомимся с библиотекой `scikit-learn`.

На занятии мы:

* исследуем связь признаков со стоимостью дома;
* обучим модель по одному, затем по трём признакам;
* сравним её прогнозы с простым прогнозом средней цены;
* оценим качество на данных, которые не участвовали в обучении.

Выполняйте ячейки по порядку. Основная часть содержит разобранные примеры, в конце — задания для самостоятельной работы.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

sns.set(style="darkgrid")

## 1. Что будем предсказывать?

Используем данные из соревнования [House Prices](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/data): продажи домов в городе Эймс, США.

* **Объект (observation)** — один дом из таблицы продаж.
* **Признаки (features)** — известные характеристики дома, например его площадь.
* **Целевая переменная (target)** — стоимость продажи, `SalePrice`, в долларах США.

Это **задача регрессии (regression)**: по признакам нужно предсказать числовую целевую переменную. Мы моделируем цены в условиях этого набора данных; для прогноза сегодняшних цен понадобятся актуальные данные.

### Почему «регрессия»?

Название связано с работами Фрэнсиса Гальтона о росте родителей и детей. В его данных средний рост детей очень высоких родителей был ближе к общему среднему росту. Это наблюдение получило название **«регрессия к среднему»**. Сегодня термин «регрессия» используется шире — для предсказания числовой целевой переменной. [Оригинальная статья Гальтона](https://galton.org/essays/1880-1889/galton-1886-jaigi-regression-stature.pdf).

In [ ]:
DATA_URL = (
    "https://www.dropbox.com/scl/fi/65u0zpc8dweb9rx1tlzol/train_sem2.csv"
    "?rlkey=5vi4mt47fs6magpldgj5wp2u9&dl=1"
)
raw_data = pd.read_csv(DATA_URL)
print(f"В таблице {raw_data.shape[0]} домов и {raw_data.shape[1]} столбец.")

В таблице много столбцов. Сегодня нам понадобятся следующие:

| Столбец | Что означает | Единицы после подготовки |
|---|---|---|
| `GrLivArea` | Надземная жилая площадь | м² |
| `GarageArea` | Площадь гаража | м² |
| `TotalBsmtSF` | Общая площадь подвала | м² |
| `LotArea` | Площадь участка | м² |
| `YearBuilt` | Год постройки | календарный год |
| `TotRmsAbvGrd` | Число надземных комнат без ванных | комнаты |
| `BedroomAbvGr` | Число надземных спален | спальни |
| `OverallQual` | Общая оценка качества материалов и отделки | баллы от 1 до 10 |
| `SalePrice` | Стоимость продажи | доллары США |

В исходном файле площади указаны в квадратных футах. Переведём четыре перечисленные площади в м²: один квадратный фут равен 0,09290304 м². Это фиксированный перевод единиц, для него не нужно изучать данные.

`Id` — идентификатор записи. Он останется в таблице, но в признаки модели его не включаем. Нулевые площади гаража и подвала могут означать отсутствие этих помещений.

In [ ]:
FEATURES = ["GrLivArea", "GarageArea", "TotalBsmtSF"]
EDA_FEATURES = FEATURES + [
    "LotArea", "YearBuilt", "TotRmsAbvGrd", "BedroomAbvGr", "OverallQual"
]

AREA_COLUMNS = FEATURES + ["LotArea"]
data = raw_data.copy()
data[AREA_COLUMNS] = data[AREA_COLUMNS] * 0.09290304

assert data[EDA_FEATURES + ["SalePrice"]].notna().all().all()
assert (data[AREA_COLUMNS] >= 0).all().all()
assert (data["SalePrice"] > 0).all()

## 2. На каких данных учимся, а на каких проверяем?

**Обучающая выборка (training set, train)** — данные, по которым подбираются коэффициенты модели.

**Валидационная выборка (validation set)** — отдельные данные для сравнения вариантов модели. Их целевые значения не передаются в `fit`.

**Тестовая выборка (test set)** — данные для окончательной проверки уже выбранного решения. Сегодня её откладываем; вернёмся к ней в конце модуля.

Разделим таблицу на **60% train, 20% validation и 20% test**. Сначала отделим 20% под test, затем четверть оставшихся 80% — под validation.

> Исследуем связи с ценой на train. Сравниваем модели на validation. Test сохраняем для окончательной проверки.

Здесь случайное разбиение имитирует новые дома из похожих условий. Для отдельной задачи «предсказать продажи следующего года» понадобилась бы проверка с учётом времени.

In [ ]:
train_val, test = train_test_split(data, test_size=0.2, random_state=42)
train, validation = train_test_split(train_val, test_size=0.25, random_state=42)

train = train.copy()
validation = validation.copy()
test = test.copy()

print(f"Train: {len(train)} домов")
print(f"Validation: {len(validation)} домов")
print(f"Test: {len(test)} домов — отложены до конца модуля")

`random_state=42` фиксирует случайное разбиение: при повторном запуске на том же файле в выборки попадут те же дома. Число 42 не имеет особого смысла. Сохраняем это разбиение во всех сравнениях.

Посмотрим на обучающую таблицу.

In [ ]:
train[EDA_FEATURES + ["SalePrice"]].head()

## 3. Исследуем данные

Сначала посмотрим, какие цены встречаются в обучающей выборке. Каждая запись — реальная продажа: высокая цена сама по себе ещё не означает ошибку в данных.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(train["SalePrice"], bins=30, color="black", ax=ax)
ax.set_xlabel("Стоимость продажи SalePrice, доллары США")
ax.set_ylabel("Количество домов")
ax.set_title("Цены домов в обучающей выборке")
plt.tight_layout()
plt.show()

### Какие признаки связаны с ценой?

Повторим **корреляцию Пирсона (Pearson correlation)** — меру направления и силы линейной связи двух числовых величин. Значения близкие к 1 или −1 означают сильную линейную связь, близкие к 0 — слабую линейную связь. Нелинейная зависимость при этом может сохраняться.

Посчитаем корреляции выбранных числовых признаков с ценой на train. Для `OverallQual` это исследовательское сравнение числовых баллов; мы пока не утверждаем, что переход между любыми соседними оценками даёт одинаковую прибавку к цене.

In [ ]:
correlations = train[EDA_FEATURES].corrwith(train["SalePrice"]).sort_values(ascending=False)

plot = sns.barplot(y=correlations.index, x=correlations.values, color="black")
plot.figure.set_size_inches(10, 5)
plot.set_xlabel("Корреляция Пирсона с SalePrice, безразмерная величина")
plot.set_ylabel("Признак")
plot.set_xlim(-1, 1)
plot.axvline(0, color="gray", linewidth=1)
plt.tight_layout()
plt.show()

Корреляция сжимает связь до одного числа. Посмотрим на сами точки.

Каждая точка на следующих графиках — один дом из train. По горизонтали отложена площадь, по вертикали — стоимость. Пока рассматриваем три площади: их смысл понятен, и в этих столбцах нет пропусков.

In [ ]:
area_labels = {
    "GrLivArea": "Надземная жилая площадь",
    "GarageArea": "Площадь гаража",
    "TotalBsmtSF": "Общая площадь подвала",
}

fig, axs = plt.subplots(figsize=(16, 5), ncols=3)
for i, feature in enumerate(["GrLivArea", "GarageArea", "TotalBsmtSF"]):
    axs[i].scatter(train[feature], train["SalePrice"], alpha=0.2, color='black')
    axs[i].set_xlabel(f"{area_labels[feature]}, м²\n{feature}")
    axs[i].set_ylabel("Стоимость продажи SalePrice, доллары США")
plt.tight_layout()
plt.show()

**Обсудим:** как меняется цена с ростом площади? Можно ли по одной площади точно определить цену? Какие дома заметно отличаются от основной группы?

### А если изменить шкалу?

Оставим те же дома, но по горизонтали возьмём квадратный корень из численного значения площади, а по вертикали — натуральный логарифм численного значения цены. `np.sqrt` вычисляет квадратный корень, `np.log` — натуральный логарифм. Площади неотрицательны, а цены положительны, поэтому эти операции здесь определены.

Преобразования сжимают большие значения. Посмотрим, как это меняет вид зависимости.

In [ ]:
fig, axs = plt.subplots(figsize=(16, 5), ncols=3)
for i, feature in enumerate(["GrLivArea", "GarageArea", "TotalBsmtSF"]):
    axs[i].scatter(np.sqrt(train[feature]), np.log(train["SalePrice"]), alpha=0.2, color='black')
    axs[i].set_xlabel(f"√({feature} / 1 м²)\n{area_labels[feature]}")
    axs[i].set_ylabel("ln(SalePrice / 1 доллар США)")
plt.tight_layout()
plt.show()

Это пока визуальное исследование. Изменение вида графика не доказывает улучшения модели: его нужно проверять на validation. **Сегодня модели обучаются на исходных площадях в м² и предсказывают цену в долларах.**

> Корреляция помогает исследовать данные, но не определяет автоматически полезность признака для модели. Слабая отдельная корреляция ещё не означает бесполезность, а сильная не гарантирует улучшения после добавления других признаков.

Помним и о связи признаков между собой: большой дом может иметь большой подвал и гараж. Отдельные корреляции с ценой не показывают самостоятельный вклад каждого из них. К устойчивости коэффициентов вернёмся на занятии про регуляризацию.

## 4. Как будем оценивать прогноз?

**Метрика качества (evaluation metric)** — число, с помощью которого оценивают предсказания модели. Сравниваем прогноз с известной стоимостью каждого дома из validation.

Основная метрика сегодня — **MAE (Mean Absolute Error), средняя абсолютная ошибка**:

$$\mathrm{MAE}=\frac{1}{n}\sum_{i=1}^{n}|y_i-\widehat y_i|.$$

**Где:** $n$ — число проверочных домов; $i$ — номер дома; $y_i$ — его фактическая стоимость; $\widehat y_i$ — прогноз стоимости; $|\cdot|$ — модуль. MAE измеряется в долларах США.

Например, MAE = 30 000 долларов означает среднюю абсолютную ошибку 30 000 долларов на проверенных домах. Ошибка отдельного дома может быть как меньше, так и больше.

Дополнительно посчитаем **RMSE (Root Mean Squared Error), корень из средней квадратичной ошибки**:

$$\mathrm{RMSE}=\sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i-\widehat y_i)^2}.$$

**Где:** $n$ — число проверочных домов; $i$ — номер дома; $y_i$ — фактическая стоимость; $\widehat y_i$ — прогноз. RMSE тоже измеряется в долларах, но сильнее реагирует на большие ошибки за счёт возведения разностей в квадрат.

Ещё одна характеристика — **MAPE (Mean Absolute Percentage Error), средняя абсолютная процентная ошибка**:

$$\mathrm{MAPE}=\frac{100}{n}\sum_{i=1}^{n}\left|\frac{y_i-\widehat y_i}{y_i}\right|.$$

**Где:** $n$ — число проверочных домов; $i$ — номер дома; $y_i$ — фактическая стоимость, не равная нулю; $\widehat y_i$ — прогноз стоимости; $|\cdot|$ — модуль; множитель 100 переводит долю в проценты.

Для каждого дома делим абсолютную ошибку на его фактическую цену, затем усредняем полученные доли. Например, MAPE = 15% означает среднюю абсолютную процентную ошибку 15% на проверенных домах. Это не предел ошибки отдельного дома.

В нашем наборе цены положительны. При нулевой фактической цене эта формула не определена; при цене, близкой к нулю, даже небольшая ошибка в долларах может дать большой вклад в MAPE. Универсального порога «хорошей MAPE» нет: приемлемая ошибка зависит от задачи.

**`mean_absolute_percentage_error` из `sklearn` возвращает долю:** например, 0.15. В коде умножаем её на 100 и получаем 15%. Библиотека подставляет малое положительное число вместо нулевого знаменателя, но это не делает обычную MAPE осмысленной для нулевых цен.

**Чем меньше MAE, RMSE и MAPE, тем лучше результат по соответствующей метрике.** Варианты модели сегодня выбираем по MAE; RMSE и MAPE используем как дополнительные характеристики.

## 5. Простой прогноз для сравнения

**Базовое решение (baseline)** — простой способ прогнозирования, с которым сравнивают более сложные модели.

Наш baseline всем домам предсказывает среднюю стоимость из train:

$$\widehat y_i=\overline y_{\mathrm{train}}=\frac{1}{n_{\mathrm{train}}}\sum_{j=1}^{n_{\mathrm{train}}}y_j.$$

**Где:** $\widehat y_i$ — прогноз для любого нового дома $i$; $\overline y_{\mathrm{train}}$ — средняя стоимость в train; $n_{\mathrm{train}}$ — число домов в train; $y_j$ — стоимость обучающего дома $j$.

Среднее минимизирует тренировочную MSE среди постоянных прогнозов. Для тренировочной MAE такой константой была бы медиана. Здесь используем среднее как понятную точку отсчёта для модели, которую тоже обучаем по квадратичной ошибке.

In [ ]:
y_train = train["SalePrice"]
y_val = validation["SalePrice"]

mean_price = y_train.mean()
baseline_pred = np.full(len(validation), mean_price)

baseline_mae = mean_absolute_error(y_val, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_val, baseline_pred))
baseline_mape = 100 * mean_absolute_percentage_error(y_val, baseline_pred)

print(f"Один прогноз для всех домов: {mean_price:,.0f} долларов США")
print(f"MAE на validation: {baseline_mae:,.0f} долларов США")
print(f"RMSE на validation: {baseline_rmse:,.0f} долларов США")
print(f"MAPE на validation: {baseline_mape:.2f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(train["GrLivArea"], y_train, alpha=0.2, color="black", label="Дома из train")
ax.axhline(mean_price, color="black", linestyle="--", label="Прогноз средней цены по train")
ax.set_xlabel("Надземная жилая площадь GrLivArea, м²")
ax.set_ylabel("Стоимость продажи SalePrice, доллары США")
ax.legend()
plt.tight_layout()
plt.show()

Горизонтальная линия означает, что прогноз вообще не учитывает площадь. **Вопрос:** для каких домов такой подход даёт особенно большие ошибки?

## 6. Первая модель: стоимость по жилой площади

**Линейная регрессия (linear regression)** предсказывает числовую целевую переменную с помощью взвешенной суммы признаков и свободного члена. Для одного признака формула имеет вид:

$$\widehat y=b+wx.$$

**Где:** $x$ — надземная жилая площадь дома в м²; $\widehat y$ — прогноз стоимости в долларах США; $w$ — коэффициент при площади, в долларах за м²; $b$ — свободный член, в долларах.

**Обучение модели (model fitting)** — подбор коэффициентов по обучающим данным. Значения $w$ и $b$ изменяются в процессе обучения; после него используем найденные значения для прогнозов.

### Немного об обучении моделей и `sklearn`

Кроме `pandas`, `numpy` и `matplotlib`, нам нужна библиотека с инструментами машинного обучения. **scikit-learn**, в коде `sklearn`, содержит готовые модели, средства подготовки данных и метрики. У многих моделей одинаковые методы `fit` и `predict`, поэтому освоенный порядок действий пригодится и дальше.

Сегодня используем класс **`LinearRegression`**. Он подбирает коэффициенты **методом наименьших квадратов (Ordinary Least Squares, OLS)**: минимизирует сумму квадратов ошибок на train. Минимум MSE находится при тех же коэффициентах, потому что деление на число обучающих объектов не меняет положение минимума:

$$\mathrm{MSE}_{\mathrm{train}}(b,w)=\frac{1}{n_{\mathrm{train}}}\sum_{i=1}^{n_{\mathrm{train}}}(y_i-b-wx_i)^2\ \longrightarrow\ \min_{b,w}.$$

**Где:** $n_{\mathrm{train}}$ — число обучающих домов; $i$ — номер дома; $x_i$ — его жилая площадь; $y_i$ — его стоимость; $b,w$ — подбираемые коэффициенты; $\min_{b,w}$ означает поиск значений этих коэффициентов, при которых выражение минимально.

На прошлом семинаре мы решали эту задачу градиентным спуском. **`LinearRegression` для нашей таблицы использует численный решатель задачи наименьших квадратов из линейной алгебры.** **Градиентный спуск здесь не используется.** Ниже выведем аналитическую формулу решения МНК и разберём условия, при которых она применима. [Описание реализации](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html).

В `sklearn` есть и другие варианты линейной регрессии — `Ridge` и `Lasso`. Они добавляют к ошибке штраф за коэффициенты; такую модификацию называют **регуляризацией (regularization)**. Её будем разбирать отдельно.

* **`fit(X_train, y_train)`** — обучить коэффициенты по признакам и известным ценам train.
* **`predict(X_new)`** — получить прогнозы по признакам домов. Их цены в этот метод не передаются.

### От градиента MSE к аналитическому решению

Повторим вывод с занятия. Пусть в train есть $n$ домов и $p$ признаков. Запишем признаки в матрицу и добавим первым столбцом единицы, чтобы учесть свободный член:

$$X=\begin{pmatrix}
1 & x_{1,1} & \cdots & x_{1,p}\\
1 & x_{2,1} & \cdots & x_{2,p}\\
\vdots & \vdots & \ddots & \vdots\\
1 & x_{n,1} & \cdots & x_{n,p}
\end{pmatrix},\qquad
\theta=\begin{pmatrix}b\\w_1\\\vdots\\w_p\end{pmatrix},\qquad
\widehat y=X\theta.$$

**Где:** $n$ — число обучающих домов; $p$ — число признаков; $x_{i,j}$ — значение признака $j$ у дома $i$; $X$ — расширенная матрица размера $n\times(p+1)$; $\theta$ — вектор из $p+1$ коэффициентов; $b$ — свободный член; $w_j$ — коэффициент признака $j$; $\widehat y$ — вектор из $n$ прогнозов.

В этом выводе **$X$ включает столбец единиц**. В таблицах для `LinearRegression` его не добавляем: библиотека учитывает свободный член сама.

### 1. Записываем MSE и её градиент

В матричной записи знакомая функция ошибки выглядит так:

$$\mathrm{MSE}(\theta)=\frac{1}{n}\|X\theta-y\|_2^2.$$

**Где:** $\theta$ — вектор коэффициентов; $X$ — расширенная матрица признаков train; $y$ — вектор из $n$ фактических цен train; $n$ — число обучающих домов; $\|X\theta-y\|_2^2$ — сумма квадратов компонентов вектора ошибок.

Возьмём частные производные по всем коэффициентам и соберём их в один вектор — **градиент (gradient)**:

$$\nabla_{\theta}\mathrm{MSE}=\frac{2}{n}X^{\mathsf T}(X\theta-y).$$

**Где:** $\nabla_{\theta}\mathrm{MSE}$ — вектор из $p+1$ частных производных; $p$ — число признаков; $n$ — число обучающих домов; $X$ — расширенная матрица; $X^{\mathsf T}$ — её транспонированная матрица размера $(p+1)\times n$; $\theta$ — коэффициенты; $y$ — фактические цены train.

### 2. Приравниваем градиент к нулю

Для этой квадратичной функции нулевой градиент соответствует минимуму MSE. Раскроем скобки и перенесём известную часть вправо:

$$\begin{aligned}
\frac{2}{n}X^{\mathsf T}(X\theta-y)&=0,\\
X^{\mathsf T}X\theta-X^{\mathsf T}y&=0,\\
(X^{\mathsf T}X)\theta&=X^{\mathsf T}y.
\end{aligned}$$

**Где:** $n$ — число обучающих домов; $X$ — расширенная матрица train; $X^{\mathsf T}$ — её транспонированная матрица; $\theta$ — неизвестные коэффициенты; $y$ — известные цены train; $0$ — нулевой вектор из $p+1$ компонентов, где $p$ — число признаков.

Последняя строка — **нормальные уравнения (normal equations)**. Это система линейных уравнений относительно коэффициентов модели.

### 3. Выражаем коэффициенты

Если столбцы $X$ линейно независимы, то матрица $X^{\mathsf T}X$ обратима. Умножим обе части нормальных уравнений **слева** на её обратную матрицу:

$$\begin{aligned}
(X^{\mathsf T}X)^{-1}(X^{\mathsf T}X)\theta
  &=(X^{\mathsf T}X)^{-1}X^{\mathsf T}y,\\
I\theta&=(X^{\mathsf T}X)^{-1}X^{\mathsf T}y,\\
\theta^*&=(X^{\mathsf T}X)^{-1}X^{\mathsf T}y.
\end{aligned}$$

**Где:** $X$ — расширенная матрица train; $X^{\mathsf T}$ — её транспонированная матрица; $(X^{\mathsf T}X)^{-1}$ — обратная матрица; $I$ — единичная матрица размера $(p+1)\times(p+1)$; $p$ — число признаков; $y$ — цены train; $\theta$ — вектор коэффициентов; $\theta^*$ — единственный вектор коэффициентов, минимизирующий MSE при указанном условии.

Это **аналитическое решение (closed-form solution)**: мы выразили коэффициенты формулой. **Градиентный спуск (gradient descent)** приближает решение последовательными шагами; результат вычислений зависит в том числе от размера шага и числа итераций. Оба подхода решают одну задачу минимизации MSE.

### Когда применима эта формула?

* **Линейная независимость столбцов** означает, что ни один столбец нельзя точно выразить линейной комбинацией остальных. Учитываем и столбец единиц. Это полный столбцовый ранг: ранг $X$ равен $p+1$. Необходимо иметь хотя бы $p+1$ строк, но одного количества строк недостаточно.
* **Статистическая независимость признаков не требуется.** Признаки могут коррелировать. Высокая корреляция сама по себе не доказывает точную линейную зависимость столбцов.
* При зависимых столбцах обратной матрицы нет и коэффициенты неединственны. При этом `LinearRegression.fit` способен найти одно из решений задачи наименьших квадратов. При почти зависимых столбцах малые изменения данных могут сильно менять коэффициенты.

При небольшом числе признаков прямые вычисления удобны: матрица $X^{\mathsf T}X$ имеет размер $(p+1)\times(p+1)$ и содержит $(p+1)^2$ элементов. Например, для трёх площадей это матрица $4\times4$. При очень большом числе признаков хранение матриц и решение системы становятся дороже. Универсальной границы «мало признаков» нет: затраты зависят также от числа строк и метода расчёта.

> Формула объясняет решение, но для численных вычислений не нужно явно получать обратную матрицу. `LinearRegression` для нашей таблицы использует решатель `scipy.linalg.lstsq`. Ниже сравним его результат с NumPy. Нормальные уравнения покажем как учебный расчёт: образование $X^{\mathsf T}X$ может усиливать ошибки округления при почти зависимых столбцах.

In [ ]:
one_feature = ["GrLivArea"]
X_train = train[one_feature]
X_val = validation[one_feature]

print("Матрица признаков train:", X_train.shape)
print("Целевая переменная train:", y_train.shape)

`X_train` — **матрица «объекты — признаки» (feature matrix)**: строки соответствуют домам, столбцы — признакам. Даже один признак передаём как таблицу с одним столбцом: поэтому используются двойные скобки через список `one_feature`.

`y_train` — одномерный набор известных стоимостей этих же домов в том же порядке.

In [ ]:
first_model = LinearRegression()
first_model.fit(X_train, y_train)

Модель обучена. Посмотрим, какие коэффициенты она нашла:

* `coef_` — массив коэффициентов при признаках;
* `intercept_` — свободный член.

По умолчанию `LinearRegression` сама учитывает свободный член. Добавлять столбец единиц к `X_train` не нужно.

In [ ]:
w = first_model.coef_[0]
b = first_model.intercept_

print(f"w = {w:,.2f} долларов США за м²")
print(f"b = {b:,.2f} долларов США")
print(f"Прогноз стоимости = {b:,.2f} + {w:,.2f} × жилая площадь в м²")

В этой модели увеличение площади на 1 м² изменяет прогноз на `w` долларов. Это описание найденной зависимости, а не обещание, что пристройка 1 м² к любому дому изменит его рыночную стоимость именно на эту сумму.

`b` задаёт высоту прямой при нулевой площади. Домов с нулевой жилой площадью в обучении нет, поэтому не будем трактовать `b` как реальную цену такого дома.

Получим прогноз для дома с жилой площадью 120 м². Сначала прямо по формуле, затем через `predict`.

In [ ]:
new_house = pd.DataFrame({"GrLivArea": [120.0]})

price_from_formula = b + w * new_house.loc[0, "GrLivArea"]
price_from_model = first_model.predict(new_house)[0]

print(f"По формуле: {price_from_formula:,.2f} долларов США")
print(f"Через predict: {price_from_model:,.2f} долларов США")

In [ ]:
pred_one = first_model.predict(X_val)

comparison = pd.DataFrame({
    "Жилая площадь, м²": validation["GrLivArea"],
    "Фактическая стоимость, $": y_val,
    "Прогноз, $": pred_one,
    "Абсолютная ошибка, $": np.abs(y_val - pred_one),
})
comparison.head()

Посмотрим на обученную прямую. Чёрные точки — train, светлые кружки с чёрной границей — validation. Прямая построена только по train; точки validation не использовались при обучении.

In [ ]:
area_grid = pd.DataFrame({
    "GrLivArea": np.linspace(train["GrLivArea"].min(), train["GrLivArea"].max(), 100)
})
line_pred = first_model.predict(area_grid)

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(train["GrLivArea"], y_train, alpha=0.2, color="black", label="Train")
ax.scatter(validation["GrLivArea"], y_val, facecolors="none", edgecolors="black",
           alpha=0.6, label="Validation")
ax.plot(area_grid["GrLivArea"], line_pred, color="black", linewidth=2,
        label="Линейная регрессия")
ax.axhline(mean_price, color="gray", linestyle="--", label="Средняя цена по train")
ax.set_xlabel("Надземная жилая площадь GrLivArea, м²")
ax.set_ylabel("Стоимость продажи SalePrice, доллары США")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Стало ли лучше?

Прямая проходит через облако точек, но визуального впечатления недостаточно. Сравним ошибки с baseline **на тех же домах из validation**.

In [ ]:
one_mae = mean_absolute_error(y_val, pred_one)
one_rmse = np.sqrt(mean_squared_error(y_val, pred_one))
one_mape = 100 * mean_absolute_percentage_error(y_val, pred_one)

results = pd.DataFrame({
    "Модель": ["Средняя цена по train", "Только жилая площадь"],
    "MAE на validation, $": [baseline_mae, one_mae],
    "RMSE на validation, $": [baseline_rmse, one_rmse],
    "MAPE на validation, %": [baseline_mape, one_mape],
})
results.round({"MAE на validation, $": 0, "RMSE на validation, $": 0, "MAPE на validation, %": 2})

### Где модель ошибается?

На левом графике сравним фактическую стоимость и прогноз. Точки на диагонали означают точное совпадение; выше неё — завышенный прогноз, ниже — заниженный.

На правом графике покажем **остатки (residuals)**:

$$e_i=y_i-\widehat y_i.$$

**Где:** $e_i$ — остаток для проверочного дома $i$, в долларах; $y_i$ — фактическая стоимость; $\widehat y_i$ — прогноз стоимости. Положительный остаток означает, что модель занизила цену, отрицательный — что завысила.

По горизонтали на правом графике будет **прогноз**, а не фактическая стоимость. Такой график помогает искать систематические ошибки. Сам по себе он не устанавливает их причину.

In [ ]:
residuals = y_val.to_numpy() - pred_one
lo = min(y_val.min(), pred_one.min())
hi = max(y_val.max(), pred_one.max())

fig, axs = plt.subplots(figsize=(13, 5), ncols=2)
axs[0].scatter(y_val, pred_one, alpha=0.2, color="black")
axs[0].plot([lo, hi], [lo, hi], color="black", linestyle="--", label="Точное совпадение")
axs[0].set_xlim(lo, hi)
axs[0].set_ylim(lo, hi)
axs[0].set_aspect("equal", adjustable="box")
axs[0].set_xlabel("Фактическая стоимость, доллары США")
axs[0].set_ylabel("Прогноз стоимости, доллары США")
axs[0].set_title("Validation: прогноз и факт")
axs[0].legend()

axs[1].scatter(pred_one, residuals, alpha=0.2, color="black")
axs[1].axhline(0, color="black", linestyle="--", label="Нулевая ошибка")
axs[1].set_xlabel("Прогноз стоимости, доллары США")
axs[1].set_ylabel("Остаток: факт − прогноз, доллары США")
axs[1].set_title("Validation: остатки")
axs[1].legend()
plt.tight_layout()
plt.show()

**Обсудим:** есть ли дома с особенно большими ошибками? Достаточно ли жилой площади, чтобы объяснить их стоимость?

Большой остаток — повод изучить объект и доступные признаки. Удаление таких домов из проверочной выборки изменило бы саму проверку. Оставляем их в сравнении.

## 8. Добавляем гараж и подвал

**Множественная линейная регрессия (multiple linear regression)** — линейная регрессия с несколькими признаками. Для трёх площадей:

$$\widehat y=b+w_1x_1+w_2x_2+w_3x_3.$$

**Где:** $\widehat y$ — прогноз стоимости в долларах; $x_1,x_2,x_3$ — жилая площадь, площадь гаража и площадь подвала в м²; $w_1,w_2,w_3$ — соответствующие коэффициенты в долларах за м²; $b$ — свободный член в долларах.

Взвешенная сумма $w_1x_1+w_2x_2+w_3x_3$ — знакомая **линейная комбинация (linear combination)** признаков: каждый признак умножается на свой коэффициент, затем произведения складываются.

Порядок действий тот же. Меняется число столбцов в матрице признаков.

In [ ]:
X_train_three = train[FEATURES]
X_val_three = validation[FEATURES]

three_model = LinearRegression()
three_model.fit(X_train_three, y_train)
pred_three = three_model.predict(X_val_three)

In [ ]:
coefficients = pd.Series(three_model.coef_, index=FEATURES, name="Коэффициент, $ за м²")
print(coefficients.round(2).to_string())
print(f"Свободный член: {three_model.intercept_:,.2f} долларов США")

Теперь коэффициент при жилой площади показывает изменение **прогноза** при увеличении жилой площади на 1 м² **при неизменных площадях гаража и подвала**. Аналогично читаются остальные коэффициенты.

Коэффициент при жилой площади может отличаться от коэффициента в первой модели: теперь при обучении учитываются ещё два связанных с ней признака.

> Размер коэффициента сам по себе не является мерой важности признака или его статистической значимости. Коэффициент описывает конкретную модель с конкретным набором признаков и единицами измерения.

### Проверим аналитическое решение через NumPy

Для тех же трёх площадей соберём расширенную матрицу `X_design`: `np.ones` создаёт столбец единиц, а `np.column_stack` соединяет его со столбцами признаков. Здесь единицы добавляем сами, потому что выполняем матричный расчёт без `fit`.

1. `np.linalg.solve` решит полученные нормальные уравнения. Перед расчётом проверим полный столбцовый ранг.
2. `np.linalg.lstsq(X_design, y, rcond=None)[0]` найдёт коэффициенты задачи наименьших квадратов непосредственно по `X_design` и `y`, без нашего вычисления `X_design.T @ X_design`. `[0]` берёт коэффициенты из результата функции; `rcond=None` оставляет стандартный численный порог определения ранга.
3. Сопоставим коэффициенты и прогнозы с уже обученной `three_model`. Возможны небольшие расхождения из-за округления.

В таблице сравниваем коэффициенты по строкам: свободный член измеряется в долларах, коэффициенты площадей — в долларах за м². `np.allclose` проверяет совпадение с допустимой численной погрешностью.

In [ ]:
X_design = np.column_stack([
    np.ones(len(X_train_three)),
    X_train_three.to_numpy(),
])
y_vector = y_train.to_numpy()

print("Размер расширенной матрицы:", X_design.shape)
print("Ранг:", np.linalg.matrix_rank(X_design))
assert np.linalg.matrix_rank(X_design) == X_design.shape[1], "Нужен полный столбцовый ранг"

# Solve the normal equations for this small, full-rank example.
theta_normal = np.linalg.solve(X_design.T @ X_design, X_design.T @ y_vector)

# Solve least squares directly, without forming the normal-equation matrix.
theta_lstsq = np.linalg.lstsq(X_design, y_vector, rcond=None)[0]
theta_sklearn = np.r_[three_model.intercept_, three_model.coef_]

coefficient_comparison = pd.DataFrame({
    "Нормальные уравнения": theta_normal,
    "NumPy lstsq": theta_lstsq,
    "sklearn": theta_sklearn,
}, index=["Свободный член"] + FEATURES)

print("Коэффициенты solve совпадают со sklearn:", np.allclose(theta_normal, theta_sklearn))
print("Коэффициенты lstsq совпадают со sklearn:", np.allclose(theta_lstsq, theta_sklearn))

X_val_design = np.column_stack([np.ones(len(X_val_three)), X_val_three.to_numpy()])
for name, theta in [("solve", theta_normal), ("lstsq", theta_lstsq)]:
    max_difference = np.max(np.abs(X_val_design @ theta - pred_three))
    print(f"Максимальное отличие прогнозов {name} от sklearn: {max_difference:.2e} долларов")

coefficient_comparison.round(6)

In [ ]:
three_mae = mean_absolute_error(y_val, pred_three)
three_rmse = np.sqrt(mean_squared_error(y_val, pred_three))
three_mape = 100 * mean_absolute_percentage_error(y_val, pred_three)

results = pd.DataFrame({
    "Модель": ["Средняя цена по train", "Только жилая площадь", "Три площади"],
    "MAE на validation, $": [baseline_mae, one_mae, three_mae],
    "RMSE на validation, $": [baseline_rmse, one_rmse, three_rmse],
    "MAPE на validation, %": [baseline_mape, one_mape, three_mape],
})
results.round({"MAE на validation, $": 0, "RMSE на validation, $": 0, "MAPE на validation, %": 2})

In [ ]:
fig, axs = plt.subplots(figsize=(16, 4), ncols=3)
for ax, metric in zip(axs, ["MAE на validation, $", "RMSE на validation, $", "MAPE на validation, %"]):
    sns.barplot(data=results, x=metric, y="Модель", color="black", ax=ax)
    ax.set_xlabel(metric.replace("$", "доллары США"))
    ax.set_ylabel("Модель")
plt.tight_layout()
plt.show()

**Вопрос:** помогли ли дополнительные признаки на validation? На сколько долларов изменилась MAE относительно модели с одной площадью?

Добавление признаков не гарантирует улучшения на новых данных. Поэтому сравниваем результаты на одной проверочной выборке и по заранее выбранной метрике.

### А какая ошибка на train?

Посчитаем те же метрики на обучающих данных. Это поможет различать качество подгонки и качество на новых объектах.

In [ ]:
pred_three_train = three_model.predict(X_train_three)

pd.DataFrame({
    "Выборка": ["Train", "Validation"],
    "MAE, доллары США": [mean_absolute_error(y_train, pred_three_train), three_mae],
    "RMSE, доллары США": [np.sqrt(mean_squared_error(y_train, pred_three_train)), three_rmse],
    "MAPE, %": [100 * mean_absolute_percentage_error(y_train, pred_three_train), three_mape],
}).round({"MAE, доллары США": 0, "RMSE, доллары США": 0, "MAPE, %": 2})

Ошибки могут различаться: в train и validation попали разные дома. Если на train ошибка мала, а на validation заметно больше, это повод проверить переобучение и различия выборок. По одному разбиению нельзя автоматически установить причину.

Ошибка на validation иногда бывает меньше ошибки на train — например, если там оказалось меньше трудных объектов. Это не противоречит тому, что коэффициенты подбирались на train.

## 9. Самостоятельная работа

### Задача 1. Проверьте прогноз вручную

Дан дом: жилая площадь 120 м², гараж 25 м², подвал 80 м².

1. Используя коэффициенты `three_model`, вычислите прогноз по формуле, без `predict`.
2. Создайте таблицу с признаками этого дома и сравните результат с `three_model.predict`.
3. На сколько изменится прогноз, если увеличить только площадь гаража на 10 м²?

Убедитесь, что порядок признаков совпадает с `FEATURES`.

In [ ]:
# TODO: calculate the prediction from the coefficients and intercept.
# TODO: compare it with predict on a one-row DataFrame.
# TODO: calculate the prediction change for 10 extra square metres of garage.

**Ваш ответ:** …

### Задача 2. Поможет ли год постройки?

К трём площадям добавьте `YearBuilt`.

1. Создайте матрицы признаков для тех же train и validation.
2. Создайте и обучите новую `LinearRegression`.
3. Получите прогнозы для validation и посчитайте MAE, RMSE и MAPE в процентах.
4. Добавьте строку в копию таблицы `results`. Сравните результат с моделью по трём площадям.

Разбиение оставьте прежним. Test в этой работе не используем.

In [ ]:
my_features = FEATURES + ["YearBuilt"]

# TODO: prepare the feature matrices, create and fit a new model.
# TODO: predict validation prices and calculate MAE, RMSE and MAPE as a percentage.
# TODO: append a row to a copy of results.

**Ваш вывод:** …

### Задача 3. Объясните результат

Напишите короткий вывод по эксперименту:

* Какой вариант вы выберете по MAE на validation? Укажите значение ошибки в долларах.
* Насколько он лучше baseline по этой метрике?
* Можно ли обещать, что для каждого нового дома ошибка будет не больше полученной MAE?
* Почему для окончательной проверки мы сохранили отдельный test?

**Ваш вывод:** …

## Что мы теперь умеем

**Данные → train / validation / test → исследование train → baseline → fit → predict → метрики на validation → сравнение.**

Мы обучили линейную модель и проверили, приносит ли она пользу по сравнению с простой средней ценой. Вывели аналитическое решение из градиента MSE, сравнили способы вычисления коэффициентов и оценили ошибки в долларах и процентах. В следующем семинаре разберём, как готовить для модели данные с пропусками и категориальными признаками.

## Источники

* [House Prices: данные и описание признаков](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/data).
* [Фрэнсис Гальтон: статья о регрессии к среднему](https://galton.org/essays/1880-1889/galton-1886-jaigi-regression-stature.pdf).
* [scikit-learn: LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html) — модель, параметры и используемый решатель.
* [scikit-learn: разделение данных и оценка моделей](https://scikit-learn.org/stable/modules/cross_validation.html#cross-validation-evaluating-estimator-performance) — роли обучающей, валидационной и тестовой выборок.
* [scikit-learn: MAE](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_absolute_error.html), [MSE](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html) и [MAPE](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_absolute_percentage_error.html).
* [NumPy: solve](https://numpy.org/doc/stable/reference/generated/numpy.linalg.solve.html) — решение квадратной системы с невырожденной матрицей.
* [NumPy: lstsq](https://numpy.org/doc/stable/reference/generated/numpy.linalg.lstsq.html) — численное решение задачи наименьших квадратов.